# Module 3: Simple Model Training on the GPU

Welcome! In this module, we will learn how to write a simple machine learning training loop on the GPU.

**Section Goals:**
* Generate a synthetic dataset directly on the GPU.
* Load model weights and biases onto the GPU.
* Train a linear regression model for 10 epochs completely on the GPU.

### Training Loops on the GPU

To train a model on the GPU, you must follow one crucial rule:

**All data and all model parameters must reside on the same device.**

If your model parameters (weights) are on the GPU, but your input data is on the CPU, PyTorch will throw a RuntimeError. Let's set up a training loop where everything is loaded on the GPU to avoid PCIe transfers during training.

### Visualizing the Training Loop

Here is a whiteboard diagram showing the loop executing entirely inside GPU memory:

![Model GPU Loop](images/linear-regression-gpu.svg)

### Step 1: Create Dataset on GPU

Let's create input features `x` and targets `y` representing the formula `y = 2 * x + 1` directly on the GPU.

In [ ]:
import torch

# Generate input numbers directly on the GPU
x = torch.linspace(1.0, 10.0, steps=100, device="cuda")
y = 2.0 * x + 1.0  # Target outputs
print("Inputs device:", x.device)

### Step 2: Initialize Parameters on GPU

Next, we initialize our model parameters (weight `w` and bias `b`) on the GPU, enabling gradient calculation.

In [ ]:
# Create weight and bias parameters directly on the GPU
w = torch.randn(1, device="cuda", requires_grad=True)
b = torch.zeros(1, device="cuda", requires_grad=True)
print(f"Initial parameters on GPU: w={w.item():.2f}, b={b.item():.2f}")

### Step 3: Run the 10-Epoch Loop

We will run a training loop for 10 epochs. Every step (forward prediction, loss calculation, backward pass, parameter update) runs entirely on the GPU.

In [ ]:
for epoch in range(10):
    pred = x * w + b  # Forward prediction
    loss = ((pred - y) ** 2).mean()  # Mean Squared Error Loss
    loss.backward()  # Backward pass (gradients)
    with torch.no_grad():
        w -= 0.05 * w.grad; b -= 0.05 * b.grad; w.grad.zero_(); b.grad.zero_()  # Update
print(f"Final w: {w.item():.2f}, Final b: {b.item():.2f}")

### Interpretation

The training loop successfully completed entirely inside the GPU VRAM. It converged toward weight `2.0` and bias `1.0` within 10 epochs. By keeping everything on the GPU, we didn't experience any PCIe transfer delays, which is the standard procedure for training deep learning models.

### Module 3 Recap

* During training, inputs and model parameters must be on the same device (GPU).
* Keeping the loop entirely inside VRAM avoids costly PCIe bus transfers.
* We can train basic models using custom PyTorch functions and parameters loaded onto CUDA.